# Week 9: Flood Risk Assessment with Cloud Data

## Research Question

**"Which areas in the Hawkesbury-Nepean region are at elevated flood risk based on terrain characteristics?"**

This notebook teaches you to:
1. Access real elevation data via cloud APIs (no manual downloads!)
2. Perform terrain analysis (slope, aspect, hillshade)
3. Classify flood risk based on multiple factors
4. Summarize results by administrative boundary

---

## Why This Matters for Your Capstone

This workflow is **directly applicable** to capstone projects. You can:
- Change the bounding box to your study area
- Adapt the risk classification for your topic (bushfire, erosion, accessibility)
- Use the same API to access satellite imagery, land cover, and more

---

## QGIS Week 4 Comparison

| What we do in Python | QGIS Week 4 equivalent |
|---------------------|------------------------|
| Download DEM via API | Download from ELVIS, add raster layer |
| `np.gradient()` for slope | Raster > Analysis > Slope |
| `np.arctan2()` for aspect | Raster > Analysis > Aspect |
| Hillshade function | Raster > Analysis > Hillshade |
| Reclassify with `np.where()` | Raster Calculator |
| `rasterstats.zonal_stats()` | Processing > Zonal Statistics |

---

# Part 1: Introduction to Cloud Data APIs

## What is Planetary Computer?

**Microsoft Planetary Computer** hosts petabytes of environmental data:
- Satellite imagery (Sentinel-2, Landsat)
- Elevation data (Copernicus DEM)
- Land cover classifications
- Climate data

## What is STAC?

**STAC** = SpatioTemporal Asset Catalog

Think of it like a library catalog for geospatial data:
- Search by location (bounding box)
- Search by time (date range)
- Search by type (satellite, DEM, etc.)

## Why use an API instead of downloading files?

| Traditional approach | API approach |
|---------------------|---------------|
| Download entire dataset (GBs) | Request only what you need |
| Files become outdated | Always access latest data |
| Manual file management | Reproducible code |
| One dataset at a time | Access hundreds of datasets |

---

## Step 0: Environment Setup

In [ ]:
# =============================================================================
# ENVIRONMENT DETECTION
# =============================================================================
# Same pattern as previous weeks - detect Colab vs local

import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing packages (2-3 minutes)...")
    
    # Core GIS packages
    !pip install geopandas rasterio rasterstats -q
    
    # Cloud data access packages
    !pip install pystac-client planetary-computer rioxarray -q
    
    print("Installation complete!")
else:
    print("Running locally")
    print("Make sure you activated: conda activate intro-gis")
    print("\nIf packages are missing, run:")
    print("  pip install pystac-client planetary-computer rioxarray")

In [ ]:
# =============================================================================
# SET UP FOLDER PATHS
# =============================================================================

from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW = Path("/content/drive/MyDrive/intro-gis/week09/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week09/data/processed")
else:
    RAW = Path("../data/raw")
    PROCESSED = Path("../data/processed")

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data folder:       {RAW.resolve()}")
print(f"Processed data folder: {PROCESSED.resolve()}")

In [ ]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
import rasterio
from rasterio.transform import from_bounds
from shapely.geometry import box

# Cloud data access
try:
    import pystac_client
    import planetary_computer
    import rioxarray
    CLOUD_AVAILABLE = True
    print("Cloud data libraries loaded successfully!")
except ImportError as e:
    CLOUD_AVAILABLE = False
    print(f"Cloud libraries not available: {e}")
    print("Will use synthetic data instead.")

print(f"\nNumPy version: {np.__version__}")

---

# Part 2: Download Elevation Data via API

We'll download elevation data from the **Copernicus DEM** - a global 30-meter resolution digital elevation model.

**QGIS equivalent:** Downloading from ELVIS (Australia) or EarthExplorer (global)

In [ ]:
# =============================================================================
# DEFINE STUDY AREA
# =============================================================================
# The Hawkesbury-Nepean region west of Sydney has significant flood risk.
# The 2022 floods in this area caused major damage.
#
# CHANGE THESE COORDINATES TO ANALYZE YOUR OWN AREA!
# Format: [west_longitude, south_latitude, east_longitude, north_latitude]

# Hawkesbury-Nepean region (flood-prone area west of Sydney)
STUDY_AREA = [150.6, -33.65, 150.85, -33.45]

# Alternative study areas you can try:
# STUDY_AREA = [150.85, -34.1, 151.1, -33.9]   # Wollongong escarpment
# STUDY_AREA = [145.0, -37.95, 145.25, -37.75] # Melbourne/Yarra
# STUDY_AREA = [153.0, -27.55, 153.2, -27.4]   # Brisbane River

# Calculate area dimensions
width_deg = STUDY_AREA[2] - STUDY_AREA[0]
height_deg = STUDY_AREA[3] - STUDY_AREA[1]
width_km = width_deg * 111 * np.cos(np.radians(STUDY_AREA[1]))
height_km = height_deg * 111

print(f"Study area bounding box: {STUDY_AREA}")
print(f"Approximate size: {width_km:.1f} km x {height_km:.1f} km")
print(f"\nThis covers the Hawkesbury-Nepean floodplain.")

In [ ]:
# =============================================================================
# CONNECT TO PLANETARY COMPUTER
# =============================================================================
# Planetary Computer uses STAC (SpatioTemporal Asset Catalog) to organize data.
# We connect to the catalog, then search for what we need.

if CLOUD_AVAILABLE:
    # Connect to the Planetary Computer STAC API
    # The 'modifier' automatically signs URLs for access
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )
    
    print("Connected to Planetary Computer!")
    print("\nAvailable DEM collections:")
    print("  - cop-dem-glo-30: Copernicus DEM (30m resolution)")
    print("  - cop-dem-glo-90: Copernicus DEM (90m resolution)")
else:
    print("Cloud connection not available - will use synthetic data")

In [ ]:
# =============================================================================
# SEARCH FOR DEM DATA COVERING OUR STUDY AREA
# =============================================================================
# The Copernicus DEM is stored in tiles. We search for tiles that cover
# our bounding box, then mosaic them together.

if CLOUD_AVAILABLE:
    print("Searching for Copernicus DEM tiles...")
    
    # Search the catalog
    search = catalog.search(
        collections=["cop-dem-glo-30"],  # 30m resolution DEM
        bbox=STUDY_AREA                   # Our study area
    )
    
    # Get list of matching items (tiles)
    items = list(search.items())
    
    print(f"Found {len(items)} DEM tile(s) covering the study area")
    
    if len(items) > 0:
        # Show what we found
        for item in items:
            print(f"  - {item.id}")
else:
    items = []

In [ ]:
# =============================================================================
# LOAD DEM DATA INTO MEMORY
# =============================================================================
# We use rioxarray to read the Cloud Optimized GeoTIFF (COG) and clip
# to our study area. This only downloads the pixels we need!

USE_REAL_DATA = False  # Will set to True if API succeeds

if CLOUD_AVAILABLE and len(items) > 0:
    try:
        print("Loading DEM from Planetary Computer...")
        print("(This downloads only the pixels within your study area)")
        
        # Get the URL for the elevation data
        item = items[0]
        dem_url = item.assets["data"].href
        
        # Open and clip to study area using rioxarray
        # This is much more efficient than downloading the whole tile!
        dem_xr = rioxarray.open_rasterio(dem_url)
        dem_xr = dem_xr.rio.clip_box(
            minx=STUDY_AREA[0],
            miny=STUDY_AREA[1],
            maxx=STUDY_AREA[2],
            maxy=STUDY_AREA[3]
        )
        
        # Convert to numpy array
        dem = dem_xr.values[0]  # Remove the band dimension
        
        # Get spatial reference info
        dem_transform = dem_xr.rio.transform()
        dem_crs = dem_xr.rio.crs
        
        # Calculate resolution
        cell_size = abs(dem_transform[0])  # Degrees
        cell_size_m = cell_size * 111000 * np.cos(np.radians(STUDY_AREA[1]))
        
        print(f"\nDEM loaded successfully!")
        print(f"  Size: {dem.shape[1]} x {dem.shape[0]} pixels")
        print(f"  Resolution: ~{cell_size_m:.0f} meters")
        print(f"  Elevation range: {np.nanmin(dem):.0f}m to {np.nanmax(dem):.0f}m")
        
        USE_REAL_DATA = True
        
    except Exception as e:
        print(f"Error loading DEM: {e}")
        print("Will use synthetic data instead.")

if not USE_REAL_DATA:
    print("Generating synthetic DEM data for demonstration...")

In [ ]:
# =============================================================================
# FALLBACK: GENERATE SYNTHETIC DEM IF API FAILS
# =============================================================================
# This creates realistic terrain with a river valley - similar to
# the Hawkesbury-Nepean floodplain.

if not USE_REAL_DATA:
    np.random.seed(42)
    
    # Create a 200x200 grid (representing ~20km x 20km at 100m resolution)
    height, width = 200, 200
    
    # Coordinate arrays
    x = np.linspace(0, 20, width)   # 0-20 km
    y = np.linspace(0, 20, height)
    X, Y = np.meshgrid(x, y)
    
    # =================================================================
    # CREATE TERRAIN FEATURES
    # =================================================================
    
    # 1. Base terrain (gentle slopes, ~50m variation)
    base = 50 + 30 * np.sin(X * 0.2) * np.cos(Y * 0.15)
    
    # 2. River valley (carved into terrain)
    # River flows roughly diagonally across the area
    river_center = 10 + 3 * np.sin(X * 0.3)
    distance_to_river = np.abs(Y - river_center)
    valley = -40 * np.exp(-distance_to_river**2 / 8)  # Deep valley near river
    
    # 3. Floodplain (flat area along river)
    floodplain = np.where(distance_to_river < 2, -20, 0)
    
    # 4. Hills on the edges
    hills = 100 * np.exp(-((X - 2)**2 + (Y - 2)**2) / 30)  # Northwest hill
    hills += 80 * np.exp(-((X - 18)**2 + (Y - 18)**2) / 25)  # Southeast hill
    
    # 5. Random micro-topography
    noise = 5 * np.random.randn(height, width)
    
    # Combine all features
    dem = base + valley + floodplain + hills + noise
    dem = np.maximum(dem, 0)  # No negative elevations
    
    # Create transform for georeferencing
    dem_transform = from_bounds(
        STUDY_AREA[0], STUDY_AREA[1],
        STUDY_AREA[2], STUDY_AREA[3],
        width, height
    )
    dem_crs = "EPSG:4326"
    cell_size_m = 100  # Assume 100m cells
    
    print("Synthetic DEM created!")
    print(f"  Size: {width} x {height} pixels")
    print(f"  Resolution: ~{cell_size_m}m")
    print(f"  Elevation range: {dem.min():.0f}m to {dem.max():.0f}m")
    print("\nNote: This is synthetic data for demonstration.")
    print("For your capstone, use real data via the API.")

In [ ]:
# =============================================================================
# VISUALIZE THE DEM
# =============================================================================
# QGIS equivalent: Add raster layer + style with terrain colormap

fig, ax = plt.subplots(figsize=(10, 8))

# Plot elevation with terrain colormap
im = ax.imshow(
    dem,
    cmap='terrain',
    extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]]
)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Elevation (m)', fontsize=11)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Digital Elevation Model - Hawkesbury-Nepean Region', fontsize=12)

# Add data source annotation
source = "Planetary Computer - Copernicus DEM" if USE_REAL_DATA else "Synthetic data"
ax.annotate(f'Source: {source}', xy=(0.02, 0.02), xycoords='axes fraction',
            fontsize=8, color='gray')

plt.tight_layout()
plt.show()

print(f"\nElevation statistics:")
print(f"  Minimum: {np.nanmin(dem):.1f} m")
print(f"  Maximum: {np.nanmax(dem):.1f} m")
print(f"  Mean:    {np.nanmean(dem):.1f} m")

---

# Part 3: Terrain Analysis

From elevation data, we derive terrain characteristics that affect flood risk:

| Derivative | What it tells us | Flood risk implication |
|------------|-----------------|------------------------|
| **Slope** | How steep is the terrain? | Flat areas = water accumulates |
| **Aspect** | Which way does water flow? | Identifies drainage patterns |
| **Hillshade** | 3D visualization | Helps identify landforms |

In [ ]:
# =============================================================================
# CALCULATE SLOPE
# =============================================================================
# Slope = rate of elevation change
# Flat areas (low slope) are more prone to flooding
#
# QGIS equivalent: Raster > Analysis > Slope

# np.gradient calculates the rate of change between adjacent cells
# Returns [dz/dy, dz/dx] - change in z for each direction
dz_dy, dz_dx = np.gradient(dem, cell_size_m)

# Slope magnitude (Pythagorean theorem)
slope_gradient = np.sqrt(dz_dx**2 + dz_dy**2)

# Convert to degrees
slope_degrees = np.degrees(np.arctan(slope_gradient))

print("Slope calculated!")
print(f"  Min slope:  {np.nanmin(slope_degrees):.1f}°")
print(f"  Max slope:  {np.nanmax(slope_degrees):.1f}°")
print(f"  Mean slope: {np.nanmean(slope_degrees):.1f}°")

print("\nSlope interpretation:")
print("  <2°:  Very flat (high flood accumulation risk)")
print("  2-5°: Gently sloping")
print("  5-15°: Moderate slopes")
print("  >15°: Steep (water runs off quickly)")

In [ ]:
# =============================================================================
# CALCULATE ASPECT
# =============================================================================
# Aspect = compass direction the slope faces
# Tells us which direction water will flow
#
# QGIS equivalent: Raster > Analysis > Aspect

# np.arctan2 calculates angle preserving quadrant information
# We use negative gradients because aspect is the downhill direction
aspect_radians = np.arctan2(-dz_dx, -dz_dy)

# Convert to degrees (0-360 compass bearing)
aspect_degrees = np.degrees(aspect_radians)
aspect_degrees = np.where(aspect_degrees < 0, aspect_degrees + 360, aspect_degrees)

print("Aspect calculated!")
print("\nAspect interpretation (compass direction):")
print("  0°/360°: North-facing")
print("  90°:     East-facing")
print("  180°:    South-facing")
print("  270°:    West-facing")

In [ ]:
# =============================================================================
# CALCULATE HILLSHADE
# =============================================================================
# Hillshade simulates sun illumination for 3D visualization
#
# QGIS equivalent: Raster > Analysis > Hillshade

def calculate_hillshade(elevation, cell_size, azimuth=315, altitude=45):
    """
    Calculate hillshade from elevation data.
    
    Parameters:
    - elevation: 2D numpy array of elevations
    - cell_size: size of each cell in same units as elevation
    - azimuth: sun direction (315° = northwest, standard)
    - altitude: sun height above horizon (45° typical)
    
    Returns:
    - hillshade: values 0-255 (0=shadow, 255=bright)
    """
    # Convert to radians
    azimuth_rad = np.radians(360 - azimuth + 90)
    altitude_rad = np.radians(altitude)
    
    # Calculate gradients
    dz_dy, dz_dx = np.gradient(elevation, cell_size)
    
    # Slope and aspect
    slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    aspect = np.arctan2(-dz_dx, dz_dy)
    
    # Hillshade formula
    hillshade = (
        np.sin(altitude_rad) * np.cos(slope) +
        np.cos(altitude_rad) * np.sin(slope) * np.cos(azimuth_rad - aspect)
    )
    
    return np.clip(hillshade, 0, 1) * 255

hillshade = calculate_hillshade(dem, cell_size_m)

print("Hillshade calculated!")
print("  Sun direction: 315° (northwest)")
print("  Sun altitude: 45°")

In [ ]:
# =============================================================================
# VISUALIZE TERRAIN ANALYSIS
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Elevation with hillshade
axes[0, 0].imshow(hillshade, cmap='gray', alpha=1,
                  extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]])
im1 = axes[0, 0].imshow(dem, cmap='terrain', alpha=0.6,
                        extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]])
axes[0, 0].set_title('Elevation with Hillshade')
plt.colorbar(im1, ax=axes[0, 0], shrink=0.7, label='Elevation (m)')

# 2. Slope
im2 = axes[0, 1].imshow(slope_degrees, cmap='YlOrRd', vmin=0, vmax=30,
                        extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]])
axes[0, 1].set_title('Slope (degrees)\nFlat areas = flood risk')
plt.colorbar(im2, ax=axes[0, 1], shrink=0.7, label='Degrees')

# 3. Aspect
im3 = axes[1, 0].imshow(aspect_degrees, cmap='hsv', vmin=0, vmax=360,
                        extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]])
axes[1, 0].set_title('Aspect (drainage direction)')
cbar3 = plt.colorbar(im3, ax=axes[1, 0], shrink=0.7)
cbar3.set_ticks([0, 90, 180, 270, 360])
cbar3.set_ticklabels(['N', 'E', 'S', 'W', 'N'])

# 4. Low-lying areas (potential flood zones)
low_threshold = np.percentile(dem[~np.isnan(dem)], 25)  # Bottom 25%
low_areas = dem <= low_threshold
im4 = axes[1, 1].imshow(dem, cmap='terrain',
                        extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]])
axes[1, 1].contour(low_areas, levels=[0.5], colors='blue', linewidths=2,
                   extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]])
axes[1, 1].set_title(f'Low-lying areas (<{low_threshold:.0f}m)\nBlue outline = potential flood zone')
plt.colorbar(im4, ax=axes[1, 1], shrink=0.7, label='Elevation (m)')

for ax in axes.flat:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle('Terrain Analysis for Flood Risk Assessment', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

# Part 4: Flood Risk Classification

We combine multiple factors to create a flood risk index:

| Factor | High Risk | Low Risk |
|--------|-----------|----------|
| Elevation | Low | High |
| Slope | Flat (<5°) | Steep (>15°) |

**QGIS equivalent:** Raster Calculator to combine layers

In [ ]:
# =============================================================================
# CREATE FLOOD RISK INDEX
# =============================================================================
# We normalize each factor to 0-1, then combine them.
# Higher score = higher flood risk
#
# QGIS equivalent: Raster Calculator with multiple inputs

# -----------------------------------------------------------------------------
# Factor 1: Elevation (lower = higher risk)
# -----------------------------------------------------------------------------
# Normalize to 0-1, then invert so low elevation = high score
dem_min = np.nanmin(dem)
dem_max = np.nanmax(dem)
elevation_score = 1 - (dem - dem_min) / (dem_max - dem_min)

print("Factor 1: Elevation")
print(f"  Low elevation ({dem_min:.0f}m) -> risk score 1.0")
print(f"  High elevation ({dem_max:.0f}m) -> risk score 0.0")

# -----------------------------------------------------------------------------
# Factor 2: Slope (flatter = higher risk)
# -----------------------------------------------------------------------------
# Flat areas allow water to accumulate
slope_max = np.nanmax(slope_degrees)
slope_score = 1 - np.clip(slope_degrees / 15, 0, 1)  # 15° or more = low risk

print("\nFactor 2: Slope")
print(f"  Flat (0°) -> risk score 1.0")
print(f"  Steep (15°+) -> risk score 0.0")

# -----------------------------------------------------------------------------
# Combine factors (weighted average)
# -----------------------------------------------------------------------------
# Weights can be adjusted based on local knowledge
WEIGHT_ELEVATION = 0.6  # Elevation is most important
WEIGHT_SLOPE = 0.4

flood_risk = (WEIGHT_ELEVATION * elevation_score + WEIGHT_SLOPE * slope_score)

print(f"\nCombined risk index (weighted):")
print(f"  {WEIGHT_ELEVATION*100:.0f}% elevation + {WEIGHT_SLOPE*100:.0f}% slope")
print(f"  Min risk score: {np.nanmin(flood_risk):.2f}")
print(f"  Max risk score: {np.nanmax(flood_risk):.2f}")

In [ ]:
# =============================================================================
# CLASSIFY INTO RISK ZONES
# =============================================================================
# Convert continuous index to categories for easier interpretation
#
# QGIS equivalent: Raster > Reclassify by Table

# Define risk thresholds
# These should be calibrated with local flood data in a real analysis
risk_classes = np.zeros_like(flood_risk, dtype=np.int8)

risk_classes[flood_risk < 0.3] = 1   # Low risk
risk_classes[(flood_risk >= 0.3) & (flood_risk < 0.5)] = 2  # Moderate
risk_classes[(flood_risk >= 0.5) & (flood_risk < 0.7)] = 3  # High
risk_classes[flood_risk >= 0.7] = 4  # Very High

# Count pixels in each class
total_pixels = np.sum(~np.isnan(flood_risk))

print("Flood Risk Classification:")
print("="*40)
for class_num, class_name in [(1, 'Low'), (2, 'Moderate'), (3, 'High'), (4, 'Very High')]:
    count = np.sum(risk_classes == class_num)
    pct = 100 * count / total_pixels
    print(f"  {class_name:12}: {count:6,} pixels ({pct:5.1f}%)")

In [ ]:
# =============================================================================
# VISUALIZE FLOOD RISK
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Continuous risk index
im1 = axes[0].imshow(flood_risk, cmap='RdYlGn_r', vmin=0, vmax=1,
                     extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]])
axes[0].set_title('Flood Risk Index (continuous)\n0 = Low risk, 1 = High risk')
plt.colorbar(im1, ax=axes[0], shrink=0.7, label='Risk Score')

# Right: Classified risk zones
cmap_risk = colors.ListedColormap(['white', 'green', 'yellow', 'orange', 'red'])
im2 = axes[1].imshow(risk_classes, cmap=cmap_risk, vmin=0, vmax=4,
                     extent=[STUDY_AREA[0], STUDY_AREA[2], STUDY_AREA[1], STUDY_AREA[3]])
axes[1].set_title('Flood Risk Zones (classified)')
cbar2 = plt.colorbar(im2, ax=axes[1], shrink=0.7)
cbar2.set_ticks([0.5, 1.5, 2.5, 3.5])
cbar2.set_ticklabels(['Low', 'Moderate', 'High', 'Very High'])

for ax in axes:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle('Flood Risk Assessment - Hawkesbury-Nepean Region', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Red/orange areas: Low-lying, flat terrain - highest flood risk")
print("- Yellow areas: Moderate risk - may experience flooding in major events")
print("- Green areas: Higher elevation or steeper slopes - lower risk")

---

# Part 5: Zonal Statistics by Administrative Boundary

To answer our research question, we need to summarize results by area:
- "What percentage of each suburb/SA2 is at high flood risk?"

**QGIS equivalent:** Processing > Zonal Statistics

In [ ]:
# =============================================================================
# CREATE ANALYSIS ZONES
# =============================================================================
# In a real analysis, you would load SA2 boundaries from ABS.
# Here we create simple quadrants for demonstration.
#
# For your capstone: Download real boundaries from ABS!

# Create quadrant zones
mid_x = (STUDY_AREA[0] + STUDY_AREA[2]) / 2
mid_y = (STUDY_AREA[1] + STUDY_AREA[3]) / 2

zones = gpd.GeoDataFrame({
    'zone_name': ['Northwest', 'Northeast', 'Southwest', 'Southeast'],
    'geometry': [
        box(STUDY_AREA[0], mid_y, mid_x, STUDY_AREA[3]),
        box(mid_x, mid_y, STUDY_AREA[2], STUDY_AREA[3]),
        box(STUDY_AREA[0], STUDY_AREA[1], mid_x, mid_y),
        box(mid_x, STUDY_AREA[1], STUDY_AREA[2], mid_y)
    ]
}, crs="EPSG:4326")

print("Analysis zones created:")
print(zones[['zone_name']])
print("\nNote: For your capstone, use real SA2/suburb boundaries!")

In [ ]:
# =============================================================================
# CALCULATE ZONAL STATISTICS
# =============================================================================
# For each zone, calculate what percentage is at high/very high risk.
#
# QGIS equivalent: Processing > Toolbox > Zonal Statistics

from rasterstats import zonal_stats

# Calculate statistics for flood risk index
stats = zonal_stats(
    zones,
    flood_risk,
    affine=dem_transform,
    stats=['mean', 'min', 'max', 'count'],
    nodata=np.nan
)

# Add to zones GeoDataFrame
zones['risk_mean'] = [s['mean'] for s in stats]
zones['risk_max'] = [s['max'] for s in stats]
zones['pixel_count'] = [s['count'] for s in stats]

# Calculate percentage of pixels at high/very high risk (class 3 or 4)
high_risk_binary = (risk_classes >= 3).astype(float)
high_risk_stats = zonal_stats(
    zones,
    high_risk_binary,
    affine=dem_transform,
    stats=['mean'],
    nodata=np.nan
)
zones['pct_high_risk'] = [s['mean'] * 100 for s in high_risk_stats]

print("Zonal Statistics - Flood Risk by Zone:")
print("="*60)
print(zones[['zone_name', 'risk_mean', 'pct_high_risk']].to_string(index=False))
print("\npct_high_risk = % of area at High or Very High flood risk")

In [ ]:
# =============================================================================
# VISUALIZE RESULTS BY ZONE
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Map showing zones colored by % high risk
zones.plot(
    column='pct_high_risk',
    cmap='Reds',
    edgecolor='black',
    linewidth=2,
    legend=True,
    legend_kwds={'label': '% at High Risk'},
    ax=axes[0]
)
axes[0].set_title('Percentage of Each Zone at High Flood Risk')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# Add zone labels
for idx, row in zones.iterrows():
    centroid = row.geometry.centroid
    axes[0].annotate(
        f"{row['zone_name']}\n{row['pct_high_risk']:.1f}%",
        xy=(centroid.x, centroid.y),
        ha='center', va='center',
        fontsize=10, fontweight='bold'
    )

# Right: Bar chart
bars = axes[1].barh(zones['zone_name'], zones['pct_high_risk'], color='coral')
axes[1].set_xlabel('% Area at High/Very High Flood Risk')
axes[1].set_title('Flood Risk by Zone')
axes[1].set_xlim(0, 100)

# Add value labels
for bar, pct in zip(bars, zones['pct_high_risk']):
    axes[1].annotate(
        f'{pct:.1f}%',
        xy=(pct + 2, bar.get_y() + bar.get_height()/2),
        va='center'
    )

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# ANSWER THE RESEARCH QUESTION
# =============================================================================

print("="*70)
print("RESEARCH QUESTION ANSWER")
print("="*70)
print()
print("Question: Which areas in the Hawkesbury-Nepean region are at elevated")
print("          flood risk based on terrain characteristics?")
print()
print("Findings:")
print("-" * 70)

# Sort by risk
zones_sorted = zones.sort_values('pct_high_risk', ascending=False)

for idx, row in zones_sorted.iterrows():
    risk_level = "HIGH PRIORITY" if row['pct_high_risk'] > 50 else "Moderate" if row['pct_high_risk'] > 25 else "Lower priority"
    print(f"  {row['zone_name']:15} - {row['pct_high_risk']:5.1f}% at high flood risk ({risk_level})")

print()
print("Methodology:")
print("  - Elevation data from Copernicus DEM (30m resolution)")
print("  - Risk factors: low elevation (60%), flat slope (40%)")
print("  - High risk defined as risk score >= 0.5")
print()
print("Limitations:")
print("  - Does not include proximity to waterways")
print("  - Does not account for drainage infrastructure")
print("  - Simplified risk model - real assessments use flood modelling")
print("="*70)

---

# Part 6: Export for QGIS

Save our Python results to use in QGIS for:
- Professional map styling
- Overlaying with other data
- Print layouts

In [ ]:
# =============================================================================
# EXPORT RASTER DATA
# =============================================================================

def save_raster(data, path, transform, crs='EPSG:4326', dtype='float32'):
    """Save a 2D numpy array as a GeoTIFF."""
    with rasterio.open(
        path, 'w', driver='GTiff',
        height=data.shape[0], width=data.shape[1],
        count=1, dtype=dtype,
        crs=crs, transform=transform
    ) as dst:
        dst.write(data.astype(dtype), 1)

# Save DEM
save_raster(dem, PROCESSED / 'dem.tif', dem_transform)
print(f"Saved: {PROCESSED / 'dem.tif'}")

# Save slope
save_raster(slope_degrees, PROCESSED / 'slope.tif', dem_transform)
print(f"Saved: {PROCESSED / 'slope.tif'}")

# Save hillshade
save_raster(hillshade, PROCESSED / 'hillshade.tif', dem_transform)
print(f"Saved: {PROCESSED / 'hillshade.tif'}")

# Save flood risk index
save_raster(flood_risk, PROCESSED / 'flood_risk_index.tif', dem_transform)
print(f"Saved: {PROCESSED / 'flood_risk_index.tif'}")

# Save classified risk zones
save_raster(risk_classes, PROCESSED / 'flood_risk_classes.tif', dem_transform, dtype='int8')
print(f"Saved: {PROCESSED / 'flood_risk_classes.tif'}")

In [ ]:
# =============================================================================
# EXPORT VECTOR DATA
# =============================================================================

# Save zones with statistics as GeoPackage
zones.to_file(PROCESSED / 'flood_risk_zones.gpkg', driver='GPKG')
print(f"Saved: {PROCESSED / 'flood_risk_zones.gpkg'}")

print("\n" + "="*50)
print("All files saved!")
print("="*50)
print(f"\nOutput folder: {PROCESSED}")
print("\nTo open in QGIS:")
print("1. Layer > Add Layer > Add Raster Layer")
print("2. Select the .tif files")
print("3. Style with appropriate colormaps")
print("\nSuggested QGIS styling:")
print("- flood_risk_classes.tif: Paletted with 4 colors (green/yellow/orange/red)")
print("- hillshade.tif: Grayscale, use as background")
print("- dem.tif: Terrain colormap")

---

# Summary: What You've Learned

## Technical Skills

| Skill | Python code | QGIS equivalent |
|-------|-------------|------------------|
| Access cloud data | `pystac_client` + `rioxarray` | Manual download |
| Calculate slope | `np.gradient()` + `np.arctan()` | Raster > Analysis > Slope |
| Calculate aspect | `np.arctan2()` | Raster > Analysis > Aspect |
| Create hillshade | Custom function | Raster > Analysis > Hillshade |
| Combine layers | Weighted average | Raster Calculator |
| Reclassify | `np.where()` | Raster > Reclassify |
| Zonal statistics | `rasterstats.zonal_stats()` | Processing > Zonal Statistics |

## Applying This to Your Capstone

1. **Change the study area**: Update `STUDY_AREA` coordinates
2. **Use real boundaries**: Download SA2/LGA boundaries from ABS
3. **Adjust the risk model**: Add more factors, change weights
4. **Try different analyses**: Bushfire risk, erosion, accessibility

## Other Planetary Computer Datasets

| Collection | Description | Use case |
|------------|-------------|----------|
| `sentinel-2-l2a` | Satellite imagery | Vegetation, land use |
| `landsat-c2-l2` | Satellite imagery | Long-term change |
| `cop-dem-glo-30` | Elevation | Terrain analysis |
| `io-lulc-9-class` | Land cover | Urban/rural mapping |

**Explore more:** [planetarycomputer.microsoft.com/catalog](https://planetarycomputer.microsoft.com/catalog)

---

## Save Your Work

- **Colab:** File > Save a copy in Drive
- **Local:** Ctrl+S (Windows) or Cmd+S (Mac)